# 🚕 Q-Learning en Taxi-v4 — Trabajo en casa

En este ejercicio aplicará Q-Learning al ambiente **Taxi-v4** de Gymnasium.

La lógica es la misma trabajada en FrozenLake:

$$
(s_t,a_t) \rightarrow (r_{t+1},s_{t+1})
$$

y la actualización:

$$
Q(s_t,a_t)
\leftarrow
Q(s_t,a_t)
+
\alpha
\left[
r_{t+1}
+
\gamma \max_a Q(s_{t+1},a)
-
Q(s_t,a_t)
\right]
$$

## Objetivo

Implementar y analizar un agente Q-Learning capaz de aprender a recoger un pasajero y llevarlo a su destino.


## 1. Preparación

Si estás trabajando en Google Colab, ejecuta la siguiente celda para instalar Gymnasium:

```bash
pip install gymnasium[toy-text]
```


In [1]:
import gymnasium as gym
import numpy as np
import random
import matplotlib.pyplot as plt

from IPython.display import HTML, display
from matplotlib import animation

SEED = 42
np.random.seed(SEED)
random.seed(SEED)


## 2. Crear el ambiente

Taxi tiene un número de estados mucho mayor que FrozenLake.

Cada estado codifica:

- posición del taxi,
- ubicación del pasajero,
- destino del pasajero.

Las acciones posibles son:

| Acción | Significado |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |


In [2]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

print("Número de estados:", env.observation_space.n)
print("Número de acciones:", env.action_space.n)


Número de estados: 500
Número de acciones: 6


### Pregunta 1

**Taxi-v4 tiene 500 estados y 6 acciones**, por lo que el espacio de acciones tiene:

$$
|\mathcal{S}|=500,\qquad |\mathcal{A}|=6
$$

Taxi tiene más estados que FrozenLake porque un estado de Taxi no representa solamente la posición del agente. También incluye:

- la posición del taxi;
- la ubicación del pasajero;
- el destino del pasajero.

Por eso existen muchas más combinaciones posibles. En Taxi-v4 hay:

$$
500 = 25\text{ posiciones del taxi}\times 5\text{ situaciones del pasajero}
\times 4\text{ destinos}
$$

mientras que FrozenLake usado normalmente en clase tiene muchos menos estados.


## 3. Observar una interacción

Ejecute una acción aleatoria y observe qué devuelve el ambiente.


In [3]:
state, info = env.reset(seed=42)
env.action_space.seed(42)

action = env.action_space.sample()

next_state, reward, terminated, truncated, info = env.step(action)

print("Estado:", state)
print("Acción:", action)
print("Nuevo estado:", next_state)
print("Recompensa:", reward)
print("Terminated:", terminated)
print("Truncated:", truncated)


Estado: 386
Acción: 0
Nuevo estado: 486
Recompensa: -1
Terminated: False
Truncated: False


### Pregunta 2

En la interacción anterior:

- $s_t$ es el **estado actual**, es decir, el valor almacenado en `state`.
- $a_t$ es la **acción elegida**, es decir, el valor almacenado en `action`.
- $r_{t+1}$ es la **recompensa obtenida después de ejecutar la acción**, es decir, `reward`.
- $s_{t+1}$ es el **nuevo estado**, es decir, `next_state`.

En forma compacta:

$$
(s_t,a_t,r_{t+1},s_{t+1})
=
(\text{state},\text{action},\text{reward},\text{next\_state})
$$

Estos cuatro elementos describen una transición del proceso de decisión de Markov: el agente estaba en un estado, tomó una acción, recibió una recompensa y pasó a otro estado.


## 4. Inicializar la Q-table

Cada fila corresponde a un estado y cada columna a una acción.

Inicialmente:

$$
Q(s,a)=0
$$


In [4]:
n_states = env.observation_space.n
n_actions = env.action_space.n

Q = np.zeros((n_states, n_actions))

print("Shape de Q:", Q.shape)
Q[:5]


Shape de Q: (500, 6)


array([[0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0.]])

### Pregunta 3

La Q-table tiene una fila por cada estado y una columna por cada acción:

$$
|\mathcal{S}|\times|\mathcal{A}|=500\times6=3000
$$

Por tanto, el agente debe aprender **3000 valores Q** en total.

Estos valores representan la utilidad esperada de realizar cada una de las 6 acciones desde cada uno de los 500 estados posibles.


## 5. Política $\epsilon$-greedy

Implemente una función que:

- con probabilidad $\epsilon$ seleccione una acción aleatoria;
- en otro caso seleccione:

$$
\arg\max_a Q(s,a)
$$

### Actividad 1
Complete la función.


In [5]:
def choose_action(Q, state, epsilon, env):
    # Exploración: elegir una acción aleatoria
    if np.random.random() < epsilon:
        return env.action_space.sample()

    # Explotación: elegir la acción con mayor valor Q
    # np.flatnonzero permite desempatar aleatoriamente cuando hay varias
    # acciones con el mismo valor máximo.
    q_values = Q[state]
    best_actions = np.flatnonzero(q_values == np.max(q_values))
    return int(np.random.choice(best_actions))


## 6. Actualización de Q

La regla de actualización es:

$$
Q(s,a)
\leftarrow
Q(s,a)
+
\alpha
\left[
r+
\gamma\max_{a'}Q(s',a')
-
Q(s,a)
\right]
$$

### Actividad 2
Complete la función.


In [6]:
def update_q(Q, state, action, reward, next_state, alpha, gamma):
    # Valor objetivo: recompensa inmediata + mejor valor futuro
    target = reward + gamma * np.max(Q[next_state])

    # Error temporal-diferencia (TD error)
    td_error = target - Q[state, action]

    # Actualización de la entrada correspondiente
    Q[state, action] += alpha * td_error


## 7. Entrenamiento

Ahora implemente el ciclo completo de Q-Learning.

En cada episodio:

1. reiniciar el ambiente;
2. escoger una acción;
3. ejecutar `env.step(action)`;
4. actualizar $Q(s,a)$;
5. mover el agente a `next_state`;
6. terminar cuando el episodio finalice.

Use inicialmente:

```python
alpha = 0.1
gamma = 0.95
epsilon = 0.1
episodes = 5000
```

### Actividad 3
Complete la función.


In [7]:
def train_q_learning(
    env,
    Q,
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1,
    max_steps=200
):
    rewards = []

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0

        for _ in range(max_steps):

            # 1. Escoger acción con política epsilon-greedy
            action = choose_action(Q, state, epsilon, env)

            # 2. Ejecutar acción
            next_state, reward, terminated, truncated, _ = env.step(action)

            # 3. Actualizar Q
            update_q(
                Q,
                state,
                action,
                reward,
                next_state,
                alpha,
                gamma
            )

            # 4. Actualizar estado y recompensa acumulada
            state = next_state
            total_reward += reward

            # 5. Terminar si el episodio finaliza
            if terminated or truncated:
                break

        rewards.append(total_reward)

    return Q, rewards


## 8. Entrenar el agente

Ejecute el entrenamiento una vez haya completado las funciones anteriores.


In [8]:
np.random.seed(SEED)
random.seed(SEED)

Q_initial = np.zeros((n_states, n_actions))

Q_trained, rewards = train_q_learning(
    env,
    Q_initial.copy(),
    episodes=5000,
    alpha=0.1,
    gamma=0.95,
    epsilon=0.1
)

print("Entrenamiento terminado.")
print("Recompensa promedio últimos 100 episodios:",
      np.mean(rewards[-100:]))
print("Mejor recompensa obtenida:", np.max(rewards))


Entrenamiento terminado.
Recompensa promedio últimos 100 episodios: 1.93
Mejor recompensa obtenida: 15


## 9. Curva de aprendizaje

Observe cómo cambia la recompensa durante el entrenamiento.


In [9]:
window = 100

moving_average = np.convolve(
    rewards,
    np.ones(window) / window,
    mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(window, len(rewards) + 1), moving_average)
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title(f"Taxi-v4 — recompensa promedio ({window} episodios)")
plt.grid(alpha=0.2)
plt.show()


### Pregunta 4

La curva esperada es ascendente: al comienzo el agente obtiene recompensas bajas porque todavía no conoce qué acciones son útiles. A medida que actualiza la Q-table, aprende rutas que permiten recoger y entregar al pasajero con menos movimientos innecesarios.

Con los parámetros:

$$
\alpha=0.1,\quad \gamma=0.95,\quad \epsilon=0.1
$$

la recompensa promedio normalmente empieza a mejorar durante los primeros cientos de episodios y suele estabilizarse progresivamente durante el entrenamiento. **El punto exacto de estabilización puede cambiar entre ejecuciones**, porque Q-Learning utiliza exploración aleatoria.

La curva no debe interpretarse como una convergencia perfecta. Una recompensa que se estabiliza indica que el agente encontró una **política razonablemente buena**, pero todavía puede cometer acciones innecesarias debido a la exploración y a las limitaciones del entrenamiento.

Por tanto:

- **¿La recompensa promedio mejora?** Sí, se espera una mejora clara respecto al comienzo.
- **¿Cuándo se estabiliza?** Aproximadamente después de varios cientos o miles de episodios; el número exacto depende de la semilla y de la ejecución.
- **¿Es convergencia perfecta?** No necesariamente. Es evidencia de aprendizaje y de una política cada vez mejor, no una garantía de optimalidad perfecta.


## 10. Reproducir un episodio

La siguiente función ejecuta una política greedy usando la Q-table aprendida y guarda los frames del episodio.


In [10]:
def play_episode(env, Q, max_steps=200, seed=None):
    state, _ = env.reset(seed=seed)

    frames = [env.render()]
    total_reward = 0

    for _ in range(max_steps):

        q_values = Q[state]
        max_q = np.max(q_values)

        best_actions = np.flatnonzero(q_values == max_q)
        action = int(np.random.choice(best_actions))

        next_state, reward, terminated, truncated, _ = env.step(action)

        frames.append(env.render())

        total_reward += reward
        state = next_state

        if terminated or truncated:
            break

    return frames, total_reward


def frames_to_video(frames, interval=500):
    fig = plt.figure(figsize=(6, 4))
    plt.axis("off")

    image = plt.imshow(frames[0])

    def update(frame):
        image.set_data(frame)
        return [image]

    anim = animation.FuncAnimation(
        fig,
        update,
        frames=frames,
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())


## 11. Comparar antes y después

Primero observe un Taxi sin entrenamiento usando una Q-table en cero.


In [11]:
frames_initial, reward_initial = play_episode(
    env,
    Q_initial,
    max_steps=50,
    seed=7
)

print("Recompensa total sin entrenamiento:", reward_initial)
frames_to_video(frames_initial, interval=500)


error: XDG_RUNTIME_DIR is invalid or not set in the environment.


Recompensa total sin entrenamiento: -104


Ahora observe el agente entrenado.


In [12]:
frames_trained, reward_trained = play_episode(
    env,
    Q_trained,
    max_steps=200,
    seed=7
)

print("Recompensa total después del entrenamiento:", reward_trained)
frames_to_video(frames_trained, interval=500)


Recompensa total después del entrenamiento: 10


### Pregunta 5

Al comparar los episodios se espera observar una diferencia importante:

- **Sin entrenamiento:** la Q-table contiene únicamente ceros. Por tanto, todas las acciones tienen inicialmente el mismo valor y la política no posee información sobre cómo llegar al pasajero ni al destino. El taxi suele realizar movimientos poco útiles y normalmente no completa la tarea dentro del límite establecido.
- **Después del entrenamiento:** los valores Q contienen información aprendida. El taxi tiende a desplazarse hacia el pasajero, ejecutar `Pickup`, dirigirse al destino y ejecutar `Dropoff`, evitando una cantidad importante de movimientos innecesarios.

La evidencia visual de que el agente aprendió es que el segundo episodio presenta una **secuencia de acciones coherente con la tarea**, mientras que el primero no tiene una estrategia aprendida.

Además, la recompensa total del agente entrenado permite complementar la evidencia visual: una recompensa menos negativa y, especialmente, la finalización de la tarea indican un comportamiento mucho mejor.


## 12. Analizar la política aprendida

Seleccione un estado cualquiera y observe los valores aprendidos para sus seis acciones.


In [13]:
state = 123

action_names = {
    0: "South",
    1: "North",
    2: "East",
    3: "West",
    4: "Pickup",
    5: "Dropoff"
}

q_values = Q_trained[state]
best_action = int(np.argmax(q_values))

print("Estado:", state)
print("Q-values:", q_values)
print("Mejor acción:", best_action, "-", action_names[best_action])
print()
print(
    f"Para el estado {state}, la acción con mayor valor Q es "
    f"{best_action} ({action_names[best_action]})."
)
print(
    "Esto significa que, según lo aprendido por el agente, esta acción "
    "tiene el mayor retorno acumulado esperado entre las seis acciones."
)


Estado: 123
Q-values: [-4.12352075  3.18222229 -4.03615461 -3.2131389  -6.23778954 -5.78855134]
Mejor acción: 1 - North

Para el estado 123, la acción con mayor valor Q es 1 (North).
Esto significa que, según lo aprendido por el agente, esta acción tiene el mayor retorno acumulado esperado entre las seis acciones.


### Pregunta 6

1. La acción con mayor valor Q es la que aparece en:

```python
np.argmax(Q_trained[state])
```

y se imprime como `Mejor acción`. Las acciones corresponden a:

| Valor | Acción |
|---|---|
| 0 | South |
| 1 | North |
| 2 | East |
| 3 | West |
| 4 | Pickup |
| 5 | Dropoff |

Por tanto, para el estado seleccionado, basta observar el número que imprime `Mejor acción` y consultar esta tabla.

2. Si una acción tiene un valor Q mayor que otra, significa que el agente estima que **tomar esa acción desde ese estado produce un retorno acumulado esperado mayor**, teniendo en cuenta las recompensas futuras.

3. No podemos interpretar $Q(s,a)$ únicamente como la recompensa inmediata porque Q-Learning incorpora también el valor de los estados futuros:

$$
Q(s,a)\leftarrow Q(s,a)+
\alpha\left[
r+\gamma\max_{a'}Q(s',a')-Q(s,a)
\right]
$$

Por eso una acción puede tener una recompensa inmediata baja o incluso negativa y, aun así, tener un Q alto si conduce a estados desde los cuales se puede obtener una buena recompensa posteriormente.


### Pregunta 7 — experimentación

Se modifica **solo un hiperparámetro**. En este caso cambiamos:

$$
\gamma: 0.95\rightarrow0.99
$$

`alpha` y `epsilon` se mantienen iguales.

La razón es que un `gamma` mayor hace que el agente dé más importancia a las recompensas futuras, algo especialmente relevante en Taxi porque para conseguir la recompensa de completar la tarea normalmente hay que realizar varias acciones antes.

### Experimento

Ejecutamos el mismo número de episodios y comparamos la recompensa promedio de los últimos 100 episodios.


In [14]:
# Experimento: cambiar solamente gamma
Q_gamma = np.zeros((n_states, n_actions))

Q_gamma, rewards_gamma = train_q_learning(
    env,
    Q_gamma,
    episodes=5000,
    alpha=0.1,      # igual al entrenamiento original
    gamma=0.99,     # único cambio
    epsilon=0.1,    # igual al entrenamiento original
    max_steps=200
)

window = 100
ma_original = np.convolve(
    rewards, np.ones(window) / window, mode="valid"
)
ma_gamma = np.convolve(
    rewards_gamma, np.ones(window) / window, mode="valid"
)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(window, len(rewards) + 1), ma_original, label="gamma = 0.95")
plt.plot(np.arange(window, len(rewards_gamma) + 1), ma_gamma, label="gamma = 0.99")
plt.xlabel("Episodio")
plt.ylabel("Recompensa promedio")
plt.title("Comparación del efecto de gamma")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print("Original (gamma=0.95):", np.mean(rewards[-100:]))
print("Nuevo (gamma=0.99):   ", np.mean(rewards_gamma[-100:]))


Original (gamma=0.95): 1.93
Nuevo (gamma=0.99):    3.22


### Respuesta de la Pregunta 7

Al aumentar $\gamma$ de **0.95 a 0.99**, el agente valora un poco más las recompensas futuras.

- **Velocidad de aprendizaje:** puede cambiar porque las recompensas futuras tienen mayor influencia en las actualizaciones. En Taxi esto puede ayudar a conectar las acciones actuales con la recompensa final, aunque no garantiza que aprenda más rápido en todas las ejecuciones.
- **Estabilidad:** normalmente puede mantenerse razonablemente estable, pero la dinámica de los valores Q cambia porque se propagan más las recompensas futuras.
- **Recompensa final:** se debe comparar directamente con la media de los últimos 100 episodios que imprime el experimento. Un valor mayor indica que el cambio fue favorable en esa ejecución.
- **Comportamiento:** con $\gamma=0.99$ el agente tiene más incentivo para realizar acciones que, aunque no den una gran recompensa inmediata, lo acerquen a completar la tarea.

La conclusión correcta debe basarse en la gráfica y en los dos valores impresos por el experimento. **No se debe afirmar que $\gamma=0.99$ siempre es mejor**, porque el resultado exacto depende de la semilla y de la aleatoriedad del entrenamiento.

En términos generales, este experimento muestra que $\gamma$ controla cuánto peso le da Q-Learning a las recompensas futuras frente a las inmediatas.
